In [1]:
from config import config
from src.data import load_data

train_df, test_df = load_data(config)

print('Train:', train_df.shape)
print('Test:', test_df.shape)

assert train_df.shape == (1460, 81)
assert test_df.shape == (1459, 80)

assert 'SalePrice' in train_df.columns
assert 'SalePrice' not in test_df.columns

assert train_df['Id'].is_unique
assert test_df['Id'].is_unique

assert train_df['SalePrice'].notna().all()
assert (train_df['SalePrice'] > 0).all()

print(train_df['MasVnrType'].value_counts(dropna=False))

Train: (1460, 81)
Test: (1459, 80)
MasVnrType
None       864
BrkFace    445
Stone      128
BrkCmn      15
NaN          8
Name: count, dtype: int64


In [2]:
import pandas as pd
from config import config

# Старый вариант
train_old = pd.read_csv(config.paths.path_to_csv)

# Новый вариант
train_new = pd.read_csv(
    config.paths.path_to_csv,
    keep_default_na=False,
    na_values=['', 'NA'],
)

comparison = pd.concat(
    [
        train_old['MasVnrType'].value_counts(dropna=False),
        train_new['MasVnrType'].value_counts(dropna=False),
    ],
    axis=1,
    keys=['Старое чтение', 'Новое чтение'],
).fillna(0).astype(int)

comparison

,Старое чтение,Новое чтение
MasVnrType,,
NaN,872,8
BrkFace,445,445
Stone,128,128
BrkCmn,15,15
None,0,864


In [3]:
train_df.loc[train_df['MasVnrType'].isna()]

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
234,235,60,RL,NaN,7851,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2010,WD,Normal,216500
529,530,20,RL,NaN,32668,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,3,2007,WD,Alloca,200624
650,651,60,FV,65.0,8125,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2008,WD,Normal,205950
936,937,20,RL,67.0,10083,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,8,2009,WD,Normal,184900
973,974,20,FV,95.0,11639,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,New,Partial,182000
977,978,120,FV,35.0,4274,Pave,Pave,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,11,2007,New,Partial,199900
1243,1244,20,RL,107.0,13891,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2006,New,Partial,465000
1278,1279,60,RL,75.0,9473,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,3,2008,WD,Normal,237000


In [4]:
import numpy as np

X = train_df.drop(columns=['Id', 'SalePrice']).copy()
y = np.log(train_df['SalePrice'])

X_test = test_df.drop(columns=['Id']).copy()

In [5]:
numerical_cols = X.select_dtypes(include='number').columns.tolist()
categorical_cols = X.select_dtypes(exclude='number').columns.tolist()

print('Числовых:', len(numerical_cols))
print('Категориальных:', len(categorical_cols))

print('Числовые признаки:')
print(numerical_cols)

Числовых: 35
Категориальных: 44
Числовые признаки:
['LotFrontage', 'LotArea', 'OverallQual', 'OverallCond', 'YearBuilt', 'YearRemodAdd', 'MasVnrArea', 'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF', '1stFlrSF', '2ndFlrSF', 'LowQualFinSF', 'GrLivArea', 'BsmtFullBath', 'BsmtHalfBath', 'FullBath', 'HalfBath', 'BedroomAbvGr', 'KitchenAbvGr', 'TotRmsAbvGrd', 'Fireplaces', 'GarageYrBlt', 'GarageCars', 'GarageArea', 'WoodDeckSF', 'OpenPorchSF', 'EnclosedPorch', '3SsnPorch', 'ScreenPorch', 'PoolArea', 'MiscVal', 'MoSold', 'YrSold']


In [6]:
from src.train_functions import build_pipeline

pipe = build_pipeline(config)

# Проверяем работоспособность на небольшой части данных.
pipe.fit(X.iloc[:100], y.iloc[:100])

predictions_log = pipe.predict(X.iloc[100:105])
predictions_dollars = np.exp(predictions_log)

assert predictions_log.shape == (5,)
assert np.isfinite(predictions_log).all()
assert (predictions_dollars > 0).all()

pd.DataFrame({
    'actual_price': train_df['SalePrice'].iloc[100:105].to_numpy(),
    'predicted_price': predictions_dollars,
})

,actual_price,predicted_price
0,205000,185897.282959
1,178000,179193.746913
2,118964,126695.245045
3,198900,203655.898993
4,169500,144172.638667


In [7]:
from src.data import split_train_holdout
from src.train_functions import cross_validate_standard
from sklearn.metrics import root_mean_squared_error

train_cv_df, holdout_df = split_train_holdout(train_df, config)

assert set(train_cv_df['Id']).isdisjoint(holdout_df['Id'])
assert len(train_cv_df) + len(holdout_df) == len(train_df)

print('Train/CV:', train_cv_df.shape)
print('Holdout:', holdout_df.shape)

scores, fold_models, oof_predictions, fold_ids = cross_validate_standard(train_cv_df, 'SalePrice', config)

assert len(oof_predictions) == len(train_cv_df)
assert np.isfinite(oof_predictions).all()
assert (fold_ids >= 0).all()

oof_rmse = root_mean_squared_error(np.log(train_cv_df['SalePrice']), oof_predictions)

print(f'\nMean CV RMSE(log): {np.mean(scores):.5f}')
print(f'CV STD: {np.std(scores):.5f}')
print(f'OOF RMSE(log): {oof_rmse:.5f}')

Train/CV: (1168, 81)
Holdout: (292, 81)
Fold 1: RMSE(log) = 0.10986
Fold 2: RMSE(log) = 0.16604
Fold 3: RMSE(log) = 0.15997
Fold 4: RMSE(log) = 0.14248
Fold 5: RMSE(log) = 0.15308

Mean CV RMSE(log): 0.14628
CV STD: 0.01982
OOF RMSE(log): 0.14762


In [8]:
for fold, pipe in enumerate(fold_models, start=1):
    preprocessor = pipe.named_steps['preprocessor']

    selected_columns = [
        column
        for name, transformer, columns in preprocessor.transformers_
        if name in ('numerical', 'categorical')
        for column in columns
    ]

    assert 'Id' not in selected_columns
    assert 'SalePrice' not in selected_columns
    assert len(selected_columns) == 79

    encoded_count = len(preprocessor.get_feature_names_out())
    print(f'Fold {fold}: исходных признаков — {len(selected_columns)}, после one-hot — {encoded_count}')

Fold 1: исходных признаков — 79, после one-hot — 312
Fold 2: исходных признаков — 79, после one-hot — 309
Fold 3: исходных признаков — 79, после one-hot — 314
Fold 4: исходных признаков — 79, после one-hot — 309
Fold 5: исходных признаков — 79, после one-hot — 308


In [9]:
from src.experiment_logging import save_oof_predictions

oof_df = save_oof_predictions(train_cv_df, oof_predictions, fold_ids, config)

assert len(oof_df) == len(train_cv_df)
assert oof_df['Id'].equals(train_cv_df['Id'])
assert np.isfinite(oof_df['prediction_price']).all()

columns = ['Id', 'SalePrice', 'prediction_price', 'error_price', 'abs_error_log']
oof_df.nlargest(10, 'abs_error_log')[columns]

OOF predictions saved: outputs\oof\1_baseline_RF.csv
OOF errors saved: outputs\oof\1_baseline_RF_errors.csv


,Id,SalePrice,prediction_price,error_price,abs_error_log
523,524,184750,528633.237557,343883.237557,1.051291
30,31,40000,96213.470662,56213.470662,0.877690
495,496,34900,82679.688823,47779.688823,0.862487
632,633,82500,192670.137286,110170.137286,0.848181
1298,1299,160000,367912.518981,207912.518981,0.832671
410,411,60000,125606.674511,65606.674511,0.738811
1349,1350,122000,254679.021093,132679.021093,0.735983
462,463,62383,128460.513438,66077.513438,0.722329
916,917,35311,71087.130609,35776.130609,0.699712
1324,1325,147000,285657.232952,138657.232952,0.664360


In [10]:
from src.train_functions import predict_with_pipeline_ensemble, calculate_regression_metrics
from src.experiment_logging import print_regression_metrics

oof_metrics = calculate_regression_metrics(train_cv_df['SalePrice'], oof_predictions)
print_regression_metrics('OOF', oof_metrics)

features_holdout = holdout_df.drop(columns=['SalePrice'])
holdout_predictions_log = predict_with_pipeline_ensemble(features_holdout, fold_models)

holdout_metrics = calculate_regression_metrics(holdout_df['SalePrice'], holdout_predictions_log)
print_regression_metrics('Holdout', holdout_metrics)


OOF:
RMSE(log): 0.14762
MAE($):    18,324.75
MSE($²):   965,909,905.65
RMSE($):   31,079.09
R²:        0.84792
MAPE:      10.32%
SMAPE:     9.80%
WAPE:      10.04%

Holdout:
RMSE(log): 0.13891
MAE($):    17,035.45
MSE($²):   769,658,563.94
RMSE($):   27,742.72
R²:        0.87328
MAPE:      9.98%
SMAPE:     9.71%
WAPE:      9.78%


In [11]:
from src.experiment_logging import save_holdout_predictions

holdout_predictions_df = save_holdout_predictions(holdout_df, holdout_predictions_log, config)

display(holdout_predictions_df.head())

Holdout predictions saved: outputs\holdout\1_baseline_RF.csv


,Id,SalePrice,actual_log,prediction_log,prediction_price,error_log,abs_error_log,error_price
1261,1262,128900,11.766792,11.804205,133813.915806,0.037413,0.037413,4913.915806
138,139,230000,12.345835,12.433006,250949.209656,0.087171,0.087171,20949.209656
578,579,146000,11.891362,12.006417,163802.511248,0.115055,0.115055,17802.511248
1125,1126,115000,11.652687,11.694981,119968.058935,0.042293,0.042293,4968.058935
1356,1357,110000,11.608236,11.682233,118448.428652,0.073997,0.073997,8448.428652


In [12]:
from src.experiment_logging import save_submission

test_predictions_log = predict_with_pipeline_ensemble(test_df, fold_models)
submission_df = save_submission(test_df, test_predictions_log, config)

print('Submission shape:', submission_df.shape)
display(submission_df.head())

Submission saved: outputs\submissions\1_baseline_RF.csv
Submission shape: (1459, 2)


,Id,SalePrice
0,1461,127066.831830
1,1462,154145.209126
2,1463,178352.283953
3,1464,181434.217230
4,1465,191029.506057
